# Study 810 — Price Delay ⏳

**Do stocks that price market news *slowly* go on to earn a premium?**

Hou & Moskowitz (2005) argue that a stock into which market-wide information diffuses
**slowly** — one whose return responds to the market with a lag — should command a
**return premium** over a stock that prices the same news promptly. Their **delay**
measure: regress a name's weekly return on the contemporaneous market plus four weekly
lags of the market over a trailing year, and read off how much of the explained variance
the *lagged* terms carry (`delay = 1 − R²_contemp-only / R²_with-lags`). Sort **long
HIGH-delay / short LOW-delay**. We test it on a liquid US cross-section
(2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`357fd262912f`); the live cells run the fast synthetic control. Survivorship:
current-membership mega-caps — magnitudes are an upper bound.*


## 1. The idea in one picture

Some stocks react to market news the moment it lands; others lag — a small, neglected, hard-to-arbitrage name may only catch up a week later. Hou & Moskowitz measure that lag as **price delay**: run the market and four weekly lags against a name's return, and see how much of the co-movement only shows up in the *lagged* terms. The thesis: **slow** names are riskier / more neglected, so they should pay a premium. Buy the laggards, sell the prompt names.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=2.45, t_nw=0.41, long_bps=36.73, short_bps=34.28, gross_sharpe=0.11)
print('long high-delay / short low-delay spread: %+.2f bps/week (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-delay book %+.2f bps vs low-delay book %+.2f bps'
      % (R['long_bps'], R['short_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long high-delay / short low-delay spread: +2.45 bps/week (NW t = +0.41)
  high-delay book +36.73 bps vs low-delay book +34.28 bps
  gross spread Sharpe (before cost): 0.11


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`knob>0`: high-delay names load on the *lagged* market **and** earn a premium) and check the detector recovers it — and that it stays *silent* on the null (`knob=0`: the lag structure is there but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from price_delay import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(knob=0.0, seed=811, n_assets=40, n_days=2000))
planted = st.synthetic_detect(data.synthetic_panel(knob=0.0018, seed=810, n_assets=40, n_days=2000))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -0.13  (should be ~0)
planted world: spread NW t = +10.83  (should light up)


## 3. The honest verdict — the famous edge does *not* show up here

On this liquid mega-cap tape the long-high-delay / short-low-delay spread is **+2.45 bps/week** with NW *t* = **+0.41** — the *right sign* (high-delay names did edge out low-delay ones) but statistically **indistinguishable from zero**: a column-permutation placebo puts the observed value at only p ≈ 0.30, and it is a coin-flip in both halves of the sample (*t* = +0.17 / +0.39). The seeded synthetic control recovers a *planted* delay premium cleanly (*t* = +10.83) and stays silent on the null, so the flat real-tape result is genuine, not a broken engine — the delay premium is a **small / illiquid / neglected-stock** phenomenon that does not survive on 50 mega-caps, every one of which is priced in milliseconds. And once you charge the weekly round-trip, even the tiny positive gross edge turns **negative** (net -0.51 bps/week at 1 bp one-way). **Signal: None**, **Tradability: Mirage**.